In [1]:
import os
os.environ['TF_CUDNN_USE_AUTOTUNE']      = '0'
os.environ['TF_XLA_FLAGS']               = '--tf_xla_auto_jit=0'
os.environ['TF_DISABLE_MKL']             = '1'
os.environ['CUDA_DEVICE_MAX_CONNECTIONS'] = '1'

import tensorflow as tf
tf.config.optimizer.set_jit(False)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU: {gpus[0].name}')

import numpy as np, pandas as pd, cv2, time
from pathlib import Path
import matplotlib.pyplot as plt
print(f'TF {tf.__version__}')

I0000 00:00:1779793351.467626     473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779793351.757058     473 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779793353.051813     473 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TF 2.21.0


W0000 00:00:1779793354.992201     473 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [2]:
DATA_ROOT = Path('/home/krish/vein_detection_task_3_training')
RAW_DIR   = DATA_ROOT / 'Data/1-Videos'
SIMP_DIR  = DATA_ROOT / 'Data/3-Simple_Annotated_Videos'
VEIN_DIR  = Path('output/vein')
CKPT_DIR  = VEIN_DIR / 'checkpoints'
PLOT_DIR  = VEIN_DIR / 'plots'
for d in [CKPT_DIR, PLOT_DIR]: d.mkdir(parents=True, exist_ok=True)

IMG_SIZE   = 384
N_CLASSES  = 2
BATCH_SIZE = 16
MEAN_TF = tf.constant([0.485,0.456,0.406], dtype=tf.float32)
STD_TF  = tf.constant([0.229,0.224,0.225], dtype=tf.float32)
MEAN_NP = np.array([0.485,0.456,0.406], np.float32)
STD_NP  = np.array([0.229,0.224,0.225], np.float32)

In [3]:
df_all  = pd.read_csv(VEIN_DIR / 'metadata.csv')
df_vein = df_all[df_all.has_vein].reset_index(drop=True)
print(f'Total: {len(df_all)} | With veins: {len(df_vein)}')
print(df_vein.groupby('video').size().rename('count').to_string())

def split_by_video(df, val_frac=0.2, seed=42):
    tr, va = [], []
    for vid in df.video.unique():
        g = df[df.video==vid].sample(frac=1, random_state=seed).reset_index(drop=True)
        n = max(1, int(len(g)*val_frac))
        va.append(g.iloc[:n]); tr.append(g.iloc[n:])
    return pd.concat(tr).reset_index(drop=True), pd.concat(va).reset_index(drop=True)

train_df, val_df = split_by_video(df_vein)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

Total: 2295 | With veins: 2037
video
202207111318_38-Perf      427
202207191643_00-Moving    723
sample_data               887
Train: 1631 | Val: 406


In [4]:
def load_sample(img_path, mask_path):
    img  = tf.cast(tf.io.decode_png(tf.io.read_file(img_path),  channels=3), tf.float32) / 255.
    img  = (img - MEAN_TF) / STD_TF
    mask = tf.squeeze(
               tf.cast(tf.io.decode_png(tf.io.read_file(mask_path), channels=1), tf.int32),
               axis=-1)
    return img, mask

def make_dataset(df, batch_size, shuffle=False, repeat=1):
    ds = tf.data.Dataset.from_tensor_slices((df.frame_path.values, df.mask_path.values))
    if shuffle: ds = ds.shuffle(len(df), reshuffle_each_iteration=True)
    ds = ds.map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    if repeat > 1: ds = ds.repeat(repeat)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

STEPS = (len(train_df) * 3) // BATCH_SIZE
train_ds = make_dataset(train_df, BATCH_SIZE, shuffle=True,  repeat=3)
val_ds   = make_dataset(val_df,   BATCH_SIZE, shuffle=False)
print(f'Steps/epoch: {STEPS} | Val batches: {len(val_ds)}')

# quick sanity check
for imgs, masks in val_ds.take(1):
    print(f'img {imgs.shape} {imgs.dtype} | mask {masks.shape} unique={set(tf.unique(tf.reshape(masks,[-1]))[0].numpy())}')

Steps/epoch: 305 | Val batches: 26
img (16, 384, 384, 3) <dtype: 'float32'> | mask (16, 384, 384) unique={np.int32(0), np.int32(1)}


In [5]:
CW_VEIN = tf.constant([0.5, 15.0], dtype=tf.float32)

def vein_dice_ce_loss(y_true, y_pred):
    smooth   = 1e-6
    y_true_i = tf.cast(y_true, tf.int32)
    y_oh     = tf.one_hot(y_true_i, N_CLASSES)
    inter    = tf.reduce_sum(y_pred * y_oh, axis=[0,1,2])
    union    = tf.reduce_sum(y_pred + y_oh, axis=[0,1,2])
    dice     = 1. - tf.reduce_mean((2.*inter+smooth)/(union+smooth))
    ce       = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    ce       = tf.reduce_mean(ce * tf.gather(CW_VEIN, y_true_i))
    return 0.5*dice + 0.5*ce

def vein_iou(y_true, y_pred):
    pc = tf.cast(tf.argmax(y_pred,-1), tf.int32)
    tc = tf.cast(y_true, tf.int32)
    tp = tf.reduce_sum(tf.cast(tf.equal(pc,1) & tf.equal(tc,1),     tf.float32))
    fp = tf.reduce_sum(tf.cast(tf.equal(pc,1) & tf.not_equal(tc,1), tf.float32))
    fn = tf.reduce_sum(tf.cast(tf.not_equal(pc,1) & tf.equal(tc,1), tf.float32))
    return tp / (tp + fp + fn + 1e-6)

def vein_dice_metric(y_true, y_pred):
    pc = tf.cast(tf.argmax(y_pred,-1)==1, tf.float32)
    tc = tf.cast(tf.cast(y_true,tf.int32)==1, tf.float32)
    return 2.*tf.reduce_sum(pc*tc)/(tf.reduce_sum(pc)+tf.reduce_sum(tc)+1e-6)

def conv_block(x, f):
    for _ in range(2):
        x = tf.keras.layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
    return x

def build_unet(n_classes=2):
    inp  = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    base = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_tensor=inp)
    s1 = base.get_layer('conv1_relu').output
    s2 = base.get_layer('conv2_block3_out').output
    s3 = base.get_layer('conv3_block4_out').output
    s4 = base.get_layer('conv4_block6_out').output
    x  = base.get_layer('conv5_block3_out').output
    for skip, f in [(s4,256),(s3,128),(s2,64),(s1,32)]:
        x = tf.keras.layers.UpSampling2D(2)(x)
        x = tf.keras.layers.Concatenate()([x, skip])
        x = conv_block(x, f)
    x = tf.keras.layers.UpSampling2D(2)(x)
    x = conv_block(x, 16)
    out = tf.keras.layers.Conv2D(n_classes, 1, activation='softmax')(x)
    return tf.keras.Model(inp, out)

model = build_unet()
print(f'Params: {sum(np.prod(v.shape) for v in model.trainable_variables)/1e6:.1f}M')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss=vein_dice_ce_loss,
              metrics=[vein_iou, vein_dice_metric],
              jit_compile=False)

Params: 32.5M


In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(CKPT_DIR / 'unet_resnet50_vein_best.keras'),
        monitor='val_vein_iou', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_vein_iou', mode='max', patience=15, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_vein_iou', mode='max',
        factor=0.5, patience=5, min_lr=1e-7, verbose=1),
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    steps_per_epoch=STEPS,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/60


E0000 00:00:1779793418.528648     473 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


 28/305 ━━━━━━━━━━━━━━━━━━━━ 14:34 3s/step - loss: 0.4711 - vein_dice_metric: 0.1430 - vein_iou: 0.0825